In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import gc
from google.colab import drive

drive.mount('/content/drive')
print("⬇️ Installing Dependencies...")
!pip install -q hydra-core>=1.3.2 omegaconf>=2.3.0
!pip install -q decord
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

print("⬇️ Downloading SAM 2 LARGE (The Heavyweight)...")
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints", exist_ok=True)
    !wget -q -O checkpoints/sam2_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

from sam2.build_sam import build_sam2_video_predictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Loading SAM 2 LARGE on {device}...")

model_cfg = "sam2_hiera_l.yaml"
checkpoint = "checkpoints/sam2_hiera_large.pt"

try:
    predictor = build_sam2_video_predictor(model_cfg, checkpoint, device=device)
    print("✅ Heavy Model Loaded. Ready to crush it.")
except Exception as e:
    print(f"❌ Error loading model: {e}")

Mounted at /content/drive
⬇️ Installing Dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 147.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
⬇️ Downloading SAM 2 LARGE (The Heavyweight)...
🚀 Loading SAM 2 LARGE on cuda...
✅ Heavy Model Loaded. Ready to crush it.


In [ ]:
import glob
import sys

base_input_dir = "/content/drive/MyDrive/EgoDex_Data/input_video/video_learning_samples"
base_output_dir = "/content/drive/MyDrive/EgoDex_Data/sam2_large_output"
MAX_FRAMES = 150

def save_tracking_video(video_path, masks, output_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    centroids = []
    frame_idx = 0
    while cap.isOpened() and frame_idx < len(masks):
        ret, frame = cap.read()
        if not ret: break

        mask = masks[frame_idx]
        if mask is not None:
            overlay = frame.copy()
            mask_uint = (mask > 0).astype(np.uint8)
            overlay[mask_uint > 0] = [0, 0, 255]
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

            M = cv2.moments(mask_uint)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                centroids.append((cX, cY))
                if len(centroids) > 1:
                    pts = np.array(centroids, np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame, [pts], False, (0, 255, 255), 3)
                cv2.circle(frame, (cX, cY), 8, (0, 255, 0), -1)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

# Processing Loop
subdirs = [f.path for f in os.scandir(base_input_dir) if f.is_dir()]

for folder_path in subdirs:
    folder_name = os.path.basename(folder_path)
    video_files = glob.glob(os.path.join(folder_path, "*.mp4"))
    if not video_files: continue

    target_video_path = video_files[0]
    video_filename = os.path.basename(target_video_path)

    save_dir = os.path.join(base_output_dir, folder_name)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, video_filename.replace(".mp4", "_heavy_track.mp4"))

    if os.path.exists(save_path):
        print(f"⏩ Skipping {folder_name} (Done)")
        continue

    print(f"\n▶️ Tracking: {folder_name}/{video_filename}")

    # Initialize these to None to prevent NameError in finally block
    inference_state = None

    try:
        inference_state = predictor.init_state(video_path=target_video_path)

        cap = cv2.VideoCapture(target_video_path)
        ret, frame0 = cap.read()
        cap.release()
        if not ret: continue
        h, w, _ = frame0.shape

        # Click Center
        points = np.array([[w // 2, h // 2]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)

        _, out_obj_ids, out_mask_logits = predictor.add_new_points(
            inference_state=inference_state,
            frame_idx=0,
            obj_id=1,
            points=points,
            labels=labels,
        )

        video_masks = []
        print("   ✨ Propagating (Heavy Model)...")
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
            if out_frame_idx >= MAX_FRAMES: break
            mask = (out_mask_logits[0] > 0.0).cpu().numpy().squeeze()
            video_masks.append(mask)

        save_tracking_video(target_video_path, video_masks, save_path)
        print(f"   ✅ Saved to: {save_path}")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
    finally:
        if 'inference_state' in locals() and inference_state is not None:
            predictor.reset_state(inference_state)
        gc.collect()
        torch.cuda.empty_cache()

print("\n🎉 HEAVY TRACKING COMPLETE!")


▶️ Tracking: open_close/1.mp4


/usr/local/lib/python3.12/dist-packages/sam2/sam2_video_predictor.py:786: UserWarning: /usr/local/lib/python3.12/dist-packages/sam2/_C.so: undefined symbol: _ZNK3c1010TensorImpl15incref_pyobjectEv

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


   ✨ Propagating (Heavy Model)...


propagate in video:  88%|████████▊ | 150/171 [00:26<00:03,  5.68it/s]


   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/sam2_large_output/open_close/1_heavy_track.mp4

▶️ Tracking: add_remove_lid/0.mp4
   ✨ Propagating (Heavy Model)...


...

   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/sam2_large_output/basic_pick_and_place/1_heavy_track.mp4

▶️ Tracking: assemble_disassemble_furniture_bench_stool/14.mp4
   ✨ Propagating (Heavy Model)...


propagate in video: 100%|██████████| 120/120 [00:21<00:00,  5.70it/s]


   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/sam2_large_output/assemble_disassemble_furniture_bench_stool/14_heavy_track.mp4

▶️ Tracking: insert_remove/3.mp4
   ✨ Propagating (Heavy Model)...


propagate in video:  74%|███████▍  | 150/203 [00:26<00:09,  5.68it/s]


   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/sam2_large_output/insert_remove/3_heavy_track.mp4

🎉 HEAVY TRACKING COMPLETE!
